In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def clean_infinity_and_extremes(df, strategy='median', z_threshold=3):
    """
    Comprehensive function to handle infinity values and extreme outliers
    """
    df_cleaned = df.copy()

    # Replace infinity values
    df_cleaned.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Handle NaN values based on strategy
    if strategy == 'median':
        df_cleaned = df_cleaned.fillna(df_cleaned.median())
    elif strategy == 'mean':
        df_cleaned = df_cleaned.fillna(df_cleaned.mean())
    elif strategy == 'drop':
        df_cleaned = df_cleaned.dropna()

    # Remove extreme outliers using Z-score
    numeric_columns = df_cleaned.select_dtypes(include=[np.number]).columns

    for col in numeric_columns:
        if df_cleaned[col].std() != 0:  # Avoid division by zero
            z_scores = np.abs((df_cleaned[col] - df_cleaned[col].mean()) / df_cleaned[col].std())
            df_cleaned = df_cleaned[z_scores < z_threshold]

    return df_cleaned

def safe_log_transform(series, shift=1):
    """
    Safe logarithmic transformation that handles negative and zero values
    """
    return np.log(series + shift)

def validate_data_quality(df):
    """
    Validate data quality and report issues
    """
    issues = {}

    # Check for infinity values
    inf_count = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
    issues['infinity_values'] = inf_count

    # Check for NaN values
    nan_count = df.isnull().sum().sum()
    issues['nan_values'] = nan_count

    # Check for zero variance columns
    numeric_df = df.select_dtypes(include=[np.number])
    zero_var_cols = [col for col in numeric_df.columns if numeric_df[col].std() == 0]
    issues['zero_variance_columns'] = zero_var_cols



In [ ]:
def safe_feature_selection(X, y, k=10, method='f_regression'):
    """
    Feature selection that handles problematic data safely
    """
    # Clean the data first
    X_clean = X.copy()
    y_clean = y.copy()

    # Remove rows with any infinity or NaN values
    mask = ~(np.isinf(X_clean).any(axis=1) | np.isnan(X_clean).any(axis=1) |
             np.isinf(y_clean) | np.isnan(y_clean))

    X_clean = X_clean[mask]
    y_clean = y_clean[mask]

    # Remove zero variance features
    feature_std = X_clean.std()
    valid_features = feature_std[feature_std > 0].index
    X_clean = X_clean[valid_features]

    # Apply feature selection
    if len(valid_features) < k:
        k = len(valid_features)

    selector = SelectKBest(score_func=f_regression, k=k)
    X_selected = selector.fit_transform(X_clean, y_clean)

    selected_features = valid_features[selector.get_support()]

    return X_selected, y_clean, selected_features, mask

def safe_scaling(X_train, X_test=None):
    """
    Safe scaling that handles infinity and extreme values
    """
    scaler = StandardScaler()

    # Replace infinity with NaN, then with median
    X_train_clean = X_train.copy()
    X_train_clean.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_train_clean.fillna(X_train_clean.median(), inplace=True)

    X_train_scaled = scaler.fit_transform(X_train_clean)

    if X_test is not None:
        X_test_clean = X_test.copy()
        X_test_clean.replace([np.inf, -np.inf], np.nan, inplace=True)
        X_test_clean.fillna(X_train_clean.median(), inplace=True)
        X_test_scaled = scaler.transform(X_test_clean)
        return X_train_scaled, X_test_scaled, scaler

    return X_train_scaled, scaler

In [ ]:
class SafeSamsungPredictionSystem:
    """
    Patched version of Samsung prediction system with infinity handling
    """
    def __init__(self, data_cleaning_strategy='median', feature_selection_k=10):
        self.data_cleaning_strategy = data_cleaning_strategy
        self.feature_selection_k = feature_selection_k
        self.model = LinearRegression()
        self.scaler = None
        self.selected_features = None
        self.is_fitted = False

    def preprocess_data(self, df, target_column):
        """
        Safe preprocessing pipeline
        """
        # Separate features and target
        X = df.drop(columns=[target_column])
        y = df[target_column]

        # Clean infinity and extreme values
        df_clean = clean_infinity_and_extremes(df, strategy=self.data_cleaning_strategy)
        X_clean = df_clean.drop(columns=[target_column])
        y_clean = df_clean[target_column]

        # Safe feature selection
        X_selected, y_selected, selected_features, valid_mask = safe_feature_selection(
            X_clean, y_clean, k=self.feature_selection_k
        )

        self.selected_features = selected_features

        return X_selected, y_selected

    def fit(self, df, target_column):
        """
        Fit the model with safe preprocessing
        """
        X, y = self.preprocess_data(df, target_column)

        # Safe scaling
        X_scaled, self.scaler = safe_scaling(pd.DataFrame(X))

        # Fit the model
        self.model.fit(X_scaled, y)
        self.is_fitted = True

        return self

    def predict(self, df):
        """
        Make predictions with error handling
        """